# Calibrating the lever arms: one constant per camera

Each camera view measures **one** head rotation exactly — the in-plane one,
which needs no rotation centre and no scale — and sees the other two only
end-on, as a shift of the head:

| camera | exact (in-plane) | along the part's axis | across it |
|---|---|---|---|
| **bottom** (ventral) | **yaw** | pitch | roll |
| **front** (frontal) | **roll** | pitch | yaw |

Both shifts scale by the **same** constant for that camera: the depth of the
landmark centroid from the joint along that camera's optical axis. One number
per camera converts both of its displacement channels into angles.

**The two cameras calibrate each other.** A camera's *lateral* channel carries
precisely the rotation the other camera measures exactly — the front camera's
lateral channel is yaw, which the bottom measures exactly, and vice versa. So
`lever_inputs()` pairs them up on its own. Pitch then comes out of **both**
cameras independently, which is a genuine cross-check rather than an
assumption.

Nothing here runs in real time: the pixel channels are what get recorded, so
the constant stays revisable and can be re-applied to old data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from kinefly2.analysis import calibrate

LOG = 'logs/260819/260819_XHCS_017_YAMV/260819_XHCS_017_YAMV_flystate.h5'   # <- your recording
log = calibrate.load_log(LOG)
print(f"{len(log['merged/seq'])} rows")

for cam in ('bottom', 'front'):
    name, v = calibrate.exact_channel(log, cam)
    if name:
        print(f'{cam:7s} measures {name.upper():5s} exactly: '
              f'std {np.degrees(np.nanstd(v)):.2f} deg')

## 1. Fit each camera's lever arm against the other camera

`check_fit` refuses recordings that cannot support a constant. A lever arm
fitted from a fly that barely moved is not a measurement but noise with units,
and it would silently rescale every angle derived from it afterwards.

In [ ]:
levers = {}
for cam, ref in (('front', 'bottom'), ('bottom', 'front')):
    shift, reference, ref_name = calibrate.lever_inputs(log, cam, reference=ref)
    fit = calibrate.fit_lever(shift, reference)
    ok, why = calibrate.check_fit(fit)
    print(f'=== {cam} lever arm, referenced to {ref}\'s exact {ref_name} ===')
    print(calibrate.report(fit, ok, why), '\n')
    if ok:
        levers[cam] = fit['lever_px']

### Other reference sources

A **rotation stage** is the only source that also validates the cameras
themselves, since it does not depend on either of them:

```python
ref_t, ref_deg = np.loadtxt('stage_angles.csv', delimiter=',', unpack=True)
reference = np.interp(log['merged/t_wall'], ref_t, np.radians(ref_deg))
fit = calibrate.fit_lever(log['front/head_d_lateral'], reference)
```

To restrict the fit to **optomotor epochs** once the arena is running:

```python
mask = calibrate.epochs_from_events(log, 'stim_on')
fit = calibrate.fit_lever(shift, reference, mask=mask)
```

**If it was rejected**, the pixel channels remain valid — only the conversion
to angles is withheld. The usual cause is too little head movement. Record
again with an optomotor stimulus, or use a rotation stage.

## 2. Pitch, measured twice and independently

This is the payoff of having both views: pitch is the axial channel of *each*
camera, through *different* lever arms. If the two agree, the whole geometric
model is confirmed. Expect the front camera to be the better of the two — its
lever arm (the head's anterior–posterior depth) is considerably longer than the
bottom camera's (the dorsoventral depth), so its pitch channel carries more
signal per degree.

In [ ]:
if len(levers) == 2:
    pitch = {cam: np.degrees(calibrate.apply_lever(log[f'{cam}/head_d_axial'], lv))
             for cam, lv in levers.items()}
    m = np.isfinite(pitch['bottom']) & np.isfinite(pitch['front'])
    r = np.corrcoef(pitch['bottom'][m], pitch['front'][m])[0, 1]
    slope = np.polyfit(pitch['front'][m], pitch['bottom'][m], 1)[0]
    print(f'pitch, bottom vs front:  r = {r:+.3f}   slope = {slope:.2f} '
          f'(1.00 = the two cameras agree)')

    t = log['merged/t_host'] - log['merged/t_host'][0]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(t, pitch['front'], lw=.6, label='front')
    ax[0].plot(t, pitch['bottom'], lw=.6, label='bottom')
    ax[0].set_xlabel('time (s)'); ax[0].set_ylabel('pitch (deg)'); ax[0].legend()
    ax[1].plot(pitch['front'][m], pitch['bottom'][m], '.', ms=1, alpha=.25)
    lim = np.nanpercentile(np.abs(pitch['front'][m]), 99)
    ax[1].plot([-lim, lim], [-lim, lim], 'r-', lw=1)
    ax[1].set_xlabel('pitch from front (deg)'); ax[1].set_ylabel('pitch from bottom (deg)')
    plt.tight_layout()
else:
    print('Need both lever arms for the cross-check; see the gates above.')

## 3. All three rotations, from whichever camera measures each best

Take each rotation from the camera that measures it *exactly* where possible:
yaw from the bottom, roll from the front. Pitch has no exact source and comes
from a lever arm either way.

In [ ]:
yaw_deg  = np.degrees(log['bottom/head_yaw'])    # exact: in-plane for a ventral view
roll_deg = np.degrees(log['front/head_roll'])    # exact: in-plane for a frontal view
pitch_deg = (np.degrees(calibrate.apply_lever(log['front/head_d_axial'], levers['front']))
             if 'front' in levers else np.full_like(yaw_deg, np.nan))

for name, v in (('yaw (bottom, exact)', yaw_deg), ('roll (front, exact)', roll_deg),
                ('pitch (front, lever)', pitch_deg)):
    if np.isfinite(v).any():
        print(f'{name:22s} std {np.nanstd(v):5.2f} deg   '
              f'range {np.nanpercentile(v,1):+6.1f} .. {np.nanpercentile(v,99):+6.1f}')

## 4. Make it permanent

Put each camera's constant in the **head:** section of its own YAML:

```yaml
# config/camera_front.yaml          # config/camera_bottom.yaml
tracker:                            tracker:
  view: frontal                       view: ventral
  head:                               head:
    tracker: landmarks                  tracker: landmarks
    lever_px: 92.4                      lever_px: 31.0
```

Kinefly2 then logs all three rotations as angles and shows degrees in the GUI.
Without it, the two lever-arm channels stay NaN rather than a guess.

**Each constant is specific to that camera, that fly, that magnification and
that landmark set.** Re-fit after moving a camera, changing a lens or
refocusing. Sanity check: `lever_px` divided by your px/mm should land near a
few tenths of a millimetre — if it comes out at several millimetres, the fit
found something other than head rotation.

Also check the **sign** once: `axis_deg` defines which way positive pitch
points. Nudge the fly's head in a known direction and, if the sign is
inverted, add 180 to `axis_deg` (or drag the axis arrow in the GUI).